# VSD Watertight Mesh Salvage (fallback for flagged knees)

Best-effort **automated salvage** of the knees that [vsd_ts_ground_truth.ipynb](vsd_ts_ground_truth.ipynb)
HARD-flagged, so they can still become GT for
[gt_per_bone_modelling.ipynb](../../../notebooks/modeling/gt_per_bone_modelling.ipynb) without manual
Slicer work. It consumes the **persisted per-bone TS masks** (`data/interim/ts_bone_masks/<vid>/`) —
no TotalSegmentator / GPU rerun.

Per flagged bone:
1. **Mask cleanup** — morphological close (bridge thin necks) → keep largest 26-connected component →
   3D fill-holes.
2. **3 mm Gaussian → zero-pad → marching cubes** (caps FOV-cut ends → watertight).
3. **trimesh repair** (merge verts, drop degenerate/dup faces, fix normals, fill holes). If still not
   watertight → **VTK fallback** (`vtkFillHolesFilter` + `vtkWindowedSincPolyDataFilter` Taubin-like
   smoothing + normals + clean), then re-repair.
4. **QC gate**: `is_watertight ∧ winding-consistent ∧ volume>0 ∧ single body ∧ volume in plausible
   range`. **Never fabricate anatomy** — bones flagged `truncated`/`missing` route straight to manual.
5. **matplotlib QC** (3D mesh + AP/LAT MIP overlay) and STL export in the `VSD_002` convention.

**Environment:** HPC digital-twin, POSIX paths, matplotlib QC (no pyvista on HPC). Mask→mesh only, so
it also runs locally on masks synced back from HPC. Deps: `vtk`, `trimesh`, `scikit-image`, `scipy`,
`SimpleITK`, `matplotlib` — all already present.

In [ ]:
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path
import numpy as np, pandas as pd, SimpleITK as sitk
from scipy import ndimage
from skimage import measure
from skimage.morphology import ball
import trimesh
import vtk
from vtk.util import numpy_support
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection  # noqa: F401  (ensures 3D toolkit loads)

# --- project root ---
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data").exists(), f"could not locate project data/ from {Path.cwd()}"

# --- inputs (produced by vsd_ts_ground_truth.ipynb) ---
MASK_DIR   = ROOT / "data/interim/ts_bone_masks"                     # <vid>/<bone>.nii.gz (raw TS masks)
GEN_REPORTS = ROOT / "data/external/ground_truth/VSD_ground_truth/_reports"   # manifest.csv
RAW_HEALTHY = ROOT / "data/raw/healthy"
Z036_EXTERNAL = ROOT / "data/external/manual_contralateral_healthy/VSD_z036_Left_manual.nrrd"

# --- outputs ---
GT_ROOT  = ROOT / "data/external/ground_truth/VSD_ground_truth"      # same convention as the generator
FIG_DIR  = ROOT / "reports/figures/vsd_gt_watertight"
WT_REPORTS = GT_ROOT / "_reports"
for d in (FIG_DIR, WT_REPORTS):
    d.mkdir(parents=True, exist_ok=True)

# --- constants (mirror the generator) ---
BONES = ["femur", "tibia", "patella", "fibula"]
BONE_COLORS = {"femur": (1.0, 0.30, 0.30), "tibia": (0.30, 0.70, 1.0),
               "patella": (1.0, 0.85, 0.20), "fibula": (0.45, 1.0, 0.45)}
SMOOTH_MM = 3.00
PAD = 2
MIN_VOXELS = 100
HU_BONE = 150
VOLUME_RANGE_MM3 = {"femur": (3e4, 6e5), "tibia": (3e4, 6e5),
                    "patella": (8e2, 6e4), "fibula": (8e2, 9e4)}

# --- salvage / QC params ---
CLOSING_RADIUS = 2          # morphological closing ball radius (voxels) to bridge thin gaps
VTK_SMOOTH_ITERS = 20       # WindowedSinc iterations (Taubin-like)
VTK_PASSBAND = 0.10         # WindowedSinc passband (smaller = smoother)
# bone-level flags that mean "anatomy is genuinely absent" -> never fabricate, route to manual
NO_FABRICATE = {"truncated", "missing"}

CASE_LIMIT = None           # None = all flagged | e.g. 1 to smoke-test

print(f"ROOT     = {ROOT}")
print(f"MASK_DIR = {MASK_DIR}")
print(f"GT_ROOT  = {GT_ROOT}")

In [ ]:
EXCLUDE = {"VSD_002_Left", "VSD_002_Right", "VSD_z050_Right", "VSD_z063_Right"}

def input_path_for(vid):
    """Re-map a volume_id to its source CT (external nrrd for z036, else raw/healthy nii.gz)."""
    if vid == "VSD_z036_Left":
        return Z036_EXTERNAL
    case_id, _ = vid.rsplit("_", 1)
    p = RAW_HEALTHY / case_id.replace("VSD_", "VSD.") / f"{vid}.nii.gz"
    return p if p.exists() else None

def bone_flags(hard_flags_str):
    """Parse a manifest hard_flags string -> {bone: set(reasons)} for reasons that name a bone."""
    out = {b: set() for b in BONES}
    for tok in str(hard_flags_str or "").split(";"):
        tok = tok.strip()
        if ":" not in tok:
            continue
        reason, target = tok.split(":", 1)
        for b in BONES:
            if b in target:
                out[b].add(reason)
    return out

# --- load the generator manifest and select HARD-flagged cases ---
manifest = pd.read_csv(GEN_REPORTS / "manifest.csv")
flagged = manifest[manifest["status"] == "FLAG"].reset_index(drop=True)
WORK = []
for r in flagged.itertuples(index=False):
    vid = r.volume_id
    if vid in EXCLUDE or not (MASK_DIR / vid).exists():
        print(f"  skip {vid}: no persisted masks" if vid not in EXCLUDE else f"  skip {vid}: excluded")
        continue
    WORK.append(dict(volume_id=vid, hard=getattr(r, "hard_flags", ""),
                     flags=bone_flags(getattr(r, "hard_flags", ""))))
if CASE_LIMIT is not None:
    WORK = WORK[:CASE_LIMIT]

print(f"\n{len(flagged)} flagged in manifest -> {len(WORK)} with masks to salvage")
for w in WORK:
    print(f"  {w['volume_id']:18s} hard={w['hard']}")

## Salvage + watertight repair helpers

`clean_mask` → `smooth_field` → `field_to_mesh` (capped) → `repair_trimesh` → `vtk_repair` (fallback)
→ `qc_mesh` (gate). `field_to_mesh` is the same watertight, world-LPS inverse of
`gt_per_bone.ipynb::voxelize_on` used by the generator.

In [ ]:
def load_mask_img(vid, bone):
    """Load a persisted per-bone mask as (binary array z,y,x, sitk image). Image carries CT geometry."""
    p = MASK_DIR / vid / f"{bone}.nii.gz"
    if not p.exists():
        return None, None
    img = sitk.ReadImage(str(p))
    return (sitk.GetArrayFromImage(img) > 0).astype(np.uint8), img


def clean_mask(binary):
    """Bridge thin necks (closing) -> keep largest 26-connected component -> fill internal holes."""
    b = binary > 0
    if b.sum() == 0:
        return b.astype(np.uint8)
    b = ndimage.binary_closing(b, structure=ball(CLOSING_RADIUS))
    lab, n = ndimage.label(b, structure=ndimage.generate_binary_structure(3, 3))
    if n > 1:
        sizes = ndimage.sum(np.ones_like(lab), lab, range(1, n + 1))
        b = lab == (int(np.argmax(sizes)) + 1)
    b = ndimage.binary_fill_holes(b)
    return b.astype(np.uint8)


def smooth_field(arr, ct_img):
    sx, sy, sz = ct_img.GetSpacing()
    sigma = (SMOOTH_MM / sz, SMOOTH_MM / sy, SMOOTH_MM / sx)   # array (z,y,x)
    return ndimage.gaussian_filter(arr.astype(np.float32), sigma=sigma)


def field_to_mesh(field, ct_img, level=0.5, pad=PAD):
    """Capped, watertight marching cubes -> trimesh in world-LPS mm (inverse of voxelize_on)."""
    if float(field.max()) < level:
        return None
    padded = np.pad(field, pad, mode="constant", constant_values=0.0)
    verts, faces, _, _ = measure.marching_cubes(padded, level=level)
    verts = verts - pad
    idx = verts[:, ::-1]
    world = np.array([ct_img.TransformContinuousIndexToPhysicalPoint(tuple(map(float, p)))
                      for p in idx], dtype=np.float64)
    return trimesh.Trimesh(vertices=world, faces=faces, process=False)


def repair_trimesh(mesh):
    """In-place trimesh cleanup: merge verts, drop degenerate/dup faces, fix normals, fill holes."""
    mesh.merge_vertices()
    mesh.update_faces(mesh.nondegenerate_faces())
    mesh.update_faces(mesh.unique_faces())
    mesh.remove_unreferenced_vertices()
    trimesh.repair.fix_normals(mesh)
    trimesh.repair.fill_holes(mesh)
    return mesh


def _tm_to_vtk(mesh):
    pts = vtk.vtkPoints()
    pts.SetData(numpy_support.numpy_to_vtk(np.ascontiguousarray(mesh.vertices, dtype=np.float64)))
    faces = np.ascontiguousarray(mesh.faces, dtype=np.int64)
    conn = np.hstack([np.full((len(faces), 1), 3, dtype=np.int64), faces]).ravel()
    cells = vtk.vtkCellArray()
    cells.SetCells(len(faces), numpy_support.numpy_to_vtkIdTypeArray(conn, deep=1))
    poly = vtk.vtkPolyData(); poly.SetPoints(pts); poly.SetPolys(cells)
    return poly


def _vtk_to_tm(poly):
    verts = numpy_support.vtk_to_numpy(poly.GetPoints().GetData())
    polys = numpy_support.vtk_to_numpy(poly.GetPolys().GetData()).reshape(-1, 4)[:, 1:]
    return trimesh.Trimesh(vertices=verts, faces=polys, process=False)


def vtk_repair(mesh):
    """VTK fallback: fill holes + WindowedSinc (Taubin-like) smoothing + consistent normals + clean."""
    fill = vtk.vtkFillHolesFilter(); fill.SetInputData(_tm_to_vtk(mesh)); fill.SetHoleSize(1e9); fill.Update()
    sm = vtk.vtkWindowedSincPolyDataFilter(); sm.SetInputConnection(fill.GetOutputPort())
    sm.SetNumberOfIterations(VTK_SMOOTH_ITERS); sm.SetPassBand(VTK_PASSBAND)
    sm.BoundarySmoothingOff(); sm.FeatureEdgeSmoothingOff(); sm.NonManifoldSmoothingOn()
    sm.NormalizeCoordinatesOn(); sm.Update()
    nrm = vtk.vtkPolyDataNormals(); nrm.SetInputConnection(sm.GetOutputPort())
    nrm.ConsistencyOn(); nrm.AutoOrientNormalsOn(); nrm.SplittingOff(); nrm.Update()
    cln = vtk.vtkCleanPolyData(); cln.SetInputConnection(nrm.GetOutputPort()); cln.Update()
    return _vtk_to_tm(cln.GetOutput())


def qc_mesh(mesh, bone):
    """QC gate: watertight, winding-consistent, single body, positive volume in plausible range."""
    wt = bool(mesh.is_watertight)
    wc = bool(mesh.is_winding_consistent)
    bodies = int(mesh.body_count)
    vol = float(abs(mesh.volume)) if wt else 0.0
    vlo, vhi = VOLUME_RANGE_MM3[bone]
    ok = wt and wc and bodies == 1 and vlo <= vol <= vhi
    return ok, dict(watertight=wt, winding=wc, bodies=bodies, volume_mm3=round(vol, 1),
                    vol_in_range=bool(vlo <= vol <= vhi))


print("salvage helpers loaded: clean_mask, smooth_field, field_to_mesh, repair_trimesh, vtk_repair, qc_mesh")

In [ ]:
def _mip(vol, axis):
    return np.max(vol, axis=axis).T


def qc_visual(vid, ct_arr, masks, meshes, save_path):
    """3D of the repaired meshes + AP/LAT MIP of the CT with the cleaned masks overlaid."""
    fig = plt.figure(figsize=(14, 5))
    ax3d = fig.add_subplot(1, 3, 1, projection="3d")
    any_mesh = False
    for b in BONES:
        mesh = meshes.get(b)
        if mesh is None or len(mesh.faces) == 0:
            continue
        any_mesh = True
        v, f = mesh.vertices, mesh.faces
        if len(f) > 6000:                                  # subsample faces for a responsive 3D view
            f = f[np.random.default_rng(0).choice(len(f), 6000, replace=False)]
        ax3d.plot_trisurf(v[:, 0], v[:, 1], v[:, 2], triangles=f,
                          color=BONE_COLORS[b], alpha=0.6, linewidth=0)
    ax3d.set_title("repaired meshes" if any_mesh else "no salvaged mesh")
    try:
        ax3d.set_box_aspect((1, 1, 1))
    except Exception:
        pass

    for k, (axis, name) in enumerate([(1, "AP"), (2, "LAT")], start=2):
        ax = fig.add_subplot(1, 3, k)
        if ct_arr is not None:
            ax.imshow(_mip(ct_arr, axis), cmap="gray", vmin=-450, vmax=1050, origin="lower")
        for b in BONES:
            m = masks.get(b)
            if m is None or m.sum() < MIN_VOXELS:
                continue
            mip = _mip(m, axis) > 0
            rgba = np.zeros((*mip.shape, 4), np.float32)
            rgba[mip, :3] = BONE_COLORS[b]; rgba[mip, 3] = 0.5
            ax.imshow(rgba, origin="lower")
        ax.set_title(f"{name} (cleaned masks)"); ax.axis("off")

    handles = [plt.Line2D([0], [0], marker="s", ls="", markersize=9, color=BONE_COLORS[b], label=b)
               for b in BONES]
    fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=8, frameon=False)
    fig.suptitle(vid, fontsize=12)
    plt.tight_layout(rect=(0, 0.04, 1, 0.95))
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=110, bbox_inches="tight")
    plt.show(); plt.close(fig)


print("qc_visual loaded")

## Per-case salvage + run

In [ ]:
def salvage_case(w):
    """Salvage every bone of one flagged case. Returns a list of per-bone result rows."""
    vid, flags = w["volume_id"], w["flags"]
    case_id, side = vid.rsplit("_", 1)

    # load persisted masks; geometry comes from the mask image (== raw CT grid)
    raw_masks, ref_img = {}, None
    for b in BONES:
        arr, img = load_mask_img(vid, b)
        raw_masks[b] = arr
        if img is not None and ref_img is None:
            ref_img = img
    if ref_img is None:
        return [dict(volume_id=vid, bone="-", status="manual", method="none", note="no_masks")]

    # CT for MIP background (optional; must share the mask grid)
    ct_arr = None
    ct_path = input_path_for(vid)
    if ct_path is not None and ct_path.exists():
        a = sitk.GetArrayFromImage(sitk.ReadImage(str(ct_path)))
        if a.shape == sitk.GetArrayFromImage(ref_img).shape:
            ct_arr = a

    cleaned, meshes, rows = {}, {}, []
    out_dir = GT_ROOT / case_id / side
    for b in BONES:
        arr = raw_masks[b]
        reasons = flags.get(b, set())
        row = dict(volume_id=vid, bone=b, method="none", status="", watertight=False,
                   bodies=0, volume_mm3=0.0, reasons=";".join(sorted(reasons)), note="")

        if arr is None or int(arr.sum()) < MIN_VOXELS:
            row.update(status="manual", note="absent_mask"); rows.append(row); continue

        cleaned[b] = clean_mask(arr)
        if reasons & NO_FABRICATE:                         # never invent truncated/missing anatomy
            row.update(status="manual", note="no_fabricate"); rows.append(row); continue

        mesh = repair_trimesh(field_to_mesh(smooth_field(cleaned[b], ref_img), ref_img))
        method = "trimesh"
        ok, qc = qc_mesh(mesh, b)
        if not ok and (not qc["watertight"] or not qc["winding"] or qc["bodies"] > 1):
            mesh = repair_trimesh(vtk_repair(mesh)); method = "vtk"
            ok, qc = qc_mesh(mesh, b)
        meshes[b] = mesh
        row.update(method=method, watertight=qc["watertight"], bodies=qc["bodies"],
                   volume_mm3=qc["volume_mm3"])
        if ok:
            out_dir.mkdir(parents=True, exist_ok=True)
            mesh.export(out_dir / f"{vid} segmentation_{b}.stl")
            row["status"] = "salvaged"
        else:
            row.update(status="manual", note="qc_fail" if qc["watertight"] else "not_watertight")
        rows.append(row)

    qc_visual(vid, ct_arr, cleaned, meshes, FIG_DIR / f"{vid}.png")
    return rows


print("salvage_case loaded")

In [ ]:
import traceback

rows_all = []
for i, w in enumerate(WORK, 1):
    print(f"[{i}/{len(WORK)}] {w['volume_id']} ...", flush=True)
    try:
        rows_all.extend(salvage_case(w))
    except Exception as exc:
        traceback.print_exc()
        rows_all.append(dict(volume_id=w["volume_id"], bone="-", status="error",
                             method="none", note=f"{type(exc).__name__}: {exc}"))

wt = pd.DataFrame(rows_all)
wt.to_csv(WT_REPORTS / "watertight_manifest.csv", index=False)
manual = wt[wt["status"].isin(["manual", "error"])]
manual.to_csv(WT_REPORTS / "manual_needed.csv", index=False)

n_salv = int((wt["status"] == "salvaged").sum())
print(f"\nbones processed : {len(wt)}")
print(f"salvaged (STL)  : {n_salv}   (via {dict(wt[wt.status=='salvaged'].method.value_counts())})")
print(f"-> manual       : {len(manual)}")
if len(manual):
    print("\nbones still needing manual work:")
    for _, r in manual.iterrows():
        print(f"  {r['volume_id']:18s} {str(r['bone']):8s} {r.get('note','')}  ({r.get('reasons','')})")

# fully-salvaged cases (all 4 bones): these are ready as GT
per_case = wt[wt.bone.isin(BONES)].groupby("volume_id")["status"].apply(lambda s: (s == "salvaged").sum())
ready = sorted(per_case[per_case == len(BONES)].index)
print(f"\nfully-salvaged cases (all 4 bones -> usable GT): {ready or 'none'}")
print(f"reports -> {WT_REPORTS}  (watertight_manifest.csv, manual_needed.csv)")
print(f"QA PNGs -> {FIG_DIR}")
wt

## Offline validation (no persisted masks required)

Exercises the salvage pipeline on synthetic defects: a fragmented rod → single watertight body, the
VTK converter/repair round-trip, and the downstream `voxelize_on` contract (`.fill()` needs the mesh
watertight). Run this to smoke-test the notebook before pointing it at real flagged masks.

In [ ]:
def _voxelize_on(mesh, ref_img):
    """Downstream convention (gt_per_bone.ipynb): solid-rasterize a world-LPS mesh onto ref grid."""
    v = np.asarray(mesh.vertices)
    idx = np.array([ref_img.TransformPhysicalPointToContinuousIndex(tuple(map(float, p))) for p in v])
    m2 = mesh.copy(); m2.vertices = idx[:, ::-1]
    pts = np.round(np.asarray(m2.voxelized(pitch=1.0).fill().points)).astype(int)
    shape = sitk.GetArrayFromImage(ref_img).shape
    out = np.zeros(shape, np.uint8)
    ok = (pts >= 0).all(1) & (pts[:, 0] < shape[0]) & (pts[:, 1] < shape[1]) & (pts[:, 2] < shape[2])
    pts = pts[ok]; out[pts[:, 0], pts[:, 1], pts[:, 2]] = 1
    return out

# synthetic CT geometry
_ct = sitk.GetImageFromArray(np.zeros((60, 70, 80), np.float32))
_ct.SetSpacing((0.86, 0.86, 0.6)); _ct.SetOrigin((-120.5, -188.0, 1275.0))

# --- fragmented rod (2 pieces, 2-voxel gap) -> clean_mask must yield 1 connected component ---
rod = np.zeros((60, 70, 80), np.uint8)
rod[10:40, 32:40, 32:40] = 1
rod[24:26, 32:40, 32:40] = 0                        # cut a gap -> 2 components
_lab, _n = ndimage.label(rod, structure=ndimage.generate_binary_structure(3, 3))
assert _n == 2, f"test rod should be fragmented, got {_n} components"
cb = clean_mask(rod)
_lab, _n = ndimage.label(cb, structure=ndimage.generate_binary_structure(3, 3))
assert _n == 1, f"clean_mask should yield 1 component, got {_n}"

mesh = repair_trimesh(field_to_mesh(smooth_field(cb, _ct), _ct))
if not mesh.is_watertight:
    mesh = repair_trimesh(vtk_repair(mesh))
ok, qc = qc_mesh(mesh, "fibula")
print(f"[salvage] watertight={qc['watertight']} bodies={qc['bodies']} volume={qc['volume_mm3']}mm^3")
assert qc["watertight"] and qc["bodies"] == 1 and qc["volume_mm3"] > 0, "fragmented rod not salvaged"

# --- VTK converter + repair round-trip on an icosphere ---
sph = trimesh.creation.icosphere(subdivisions=3, radius=6.0)
back = _vtk_to_tm(_tm_to_vtk(sph))
assert len(back.faces) == len(sph.faces), "vtk<->trimesh face count changed"
rep = repair_trimesh(vtk_repair(sph))
assert rep.is_watertight, "vtk_repair should keep the sphere watertight"

# --- downstream contract: salvaged mesh solid-voxelizes to a non-empty volume ---
vox = _voxelize_on(mesh, _ct)
assert int(vox.sum()) > MIN_VOXELS, "salvaged mesh failed downstream solid voxelization"
print(f"[downstream] voxelize_on(salvaged) -> {int(vox.sum())} voxels")
print("OFFLINE VALIDATION PASS: fragmented mask salvaged to a single watertight body; "
      "VTK repair OK; downstream .fill() voxelization OK")